In [ ]:
import shap
import torch
import torch.nn as nn
import numpy as np

import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

import seaborn as sns

from sklearn.cluster import KMeans
import torch

from sklearn.decomposition import PCA
import networkx as nx
from torch_geometric.utils import to_networkx

from torch_geometric.explain import Explainer, GNNExplainer
from torch_geometric.loader import DataLoader

from torch_geometric.explain.config import ModelConfig
from scipy.stats import spearmanr
from sklearn.metrics import mean_squared_error

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data():
    """
    Loads (140,4) data from 'Feature_CNN1.txt'.
    """
    data_branch1 = []
    current_array = []
    with open('Feature_CNN1.txt', 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 140, 4)
    return X_branch1

def load_reaction_rates():
    with open('HT1-1_indel_frequency_value_percentage.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)
    
########################################
# 2. Graph Data Utilities (for GNN)
########################################

# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def RNA_reverse_complement(RNA):
    complement = {'A': 'U', 'C': 'G', 'G': 'C', 'U': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(RNA))

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))

########################################
# 2. CNN1 Branch
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (140,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*70, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,70)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,70)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import NNConv, global_mean_pool, global_max_pool


########################################
# 3. Final Fusion Model
########################################

class CNN1_Only(nn.Module):
    """
    End-to-end: 
      - CNNBranch1 => feat_cnn1
    dropout => final FC => 1
    """
    def __init__(self,
                 filters1, kernel_size1, dense_units1,  # CNN1
                 final_fc_dim,
                 dropout_rate=0.0):  # new hyperparameter for dropout
        super().__init__()
        
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)

        
        # total dimension = (dense_units1)
        total_dim = dense_units1
        
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(total_dim, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat_cnn1 = self.cnn_branch1(x1)                   # (batch, dense_units1)

        feat_cnn1 = F.dropout(feat_cnn1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(feat_cnn1))   # (batch, final_fc_dim)
        out = self.out(x)                 # (batch, 1)
        return out.view(-1)

########################################
# 4. Hybrid Dataset
########################################

class IndivDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)
        
        assert len(X1) == self.num_samples
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)  
        y = torch.tensor(self.reaction_rates[idx], dtype=torch.float)
        return x1, y

def collate(batch):
    from torch_geometric.data import Batch
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y



class FullModelWrapperForCNN1(nn.Module):
    def __init__(self, full_model):
        super().__init__()
        self.full_model = full_model.eval()

    def forward(self, x1):
        batch_size = x1.shape[0]
        output = self.full_model(x1)
        return output.view(-1, 1)  # Shape required by SHAP


def filter_extreme_by_column(matrix, threshold=0.2):
    """
    For each column in the matrix:
    - If the range (max - min) exceeds the threshold,
    - Keep only the max and min values.
    - Set other values to 0.
    """
    filtered = np.zeros_like(matrix)
    for col in range(matrix.shape[1]):
        col_vals = matrix[:, col]
        col_max = np.max(col_vals)
        col_min = np.min(col_vals)
        if (col_max - col_min) >= threshold:
            max_idx = np.argmax(col_vals)
            min_idx = np.argmin(col_vals)
            filtered[max_idx, col] = col_max
            filtered[min_idx, col] = col_min
    return filtered


import torch
from tqdm import tqdm
from collections import defaultdict
import pandas as pd
import seaborn as sns
import numpy as np
import pandas as pd


# Segment ranges: (start, end)
segments_cnn1 = [(0, 40), (40, 140)]

np.random.seed(42)
torch.manual_seed(42)


# === Example Execution ===
if __name__ == "__main__":
    # Load your trained model
    full_model = torch.load(
    "CNN1_trial_11.pt",
    weights_only=False
)
    full_model.eval()

    from sklearn.cluster import KMeans
    from sklearn.metrics import pairwise_distances_argmin_min
    import torch
    import numpy as np
    
    # Load original data
    X1_all = load_branch1_data()
    rates = load_reaction_rates()
    
    # Flatten for clustering
    X1_flat = X1_all.reshape(X1_all.shape[0], -1)
    
    # Run KMeans
    kmeans1_sample = KMeans(n_clusters=1000, random_state=42).fit(X1_flat)
    kmeans1_background = KMeans(n_clusters=100, random_state=42).fit(X1_flat)
    
    # Get indices of closest original points to each centroid
    closest_X1_indices_sample, _ = pairwise_distances_argmin_min(kmeans1_sample.cluster_centers_, X1_flat)
    closest_X1_indices_background, _ = pairwise_distances_argmin_min(kmeans1_background.cluster_centers_, X1_flat)
    
    # Get samples 
    X1_selected = X1_all[closest_X1_indices_sample]

    # Get backgrounds
    background_cnn1 = X1_all[closest_X1_indices_background]
    
    # Use these indices to extract the corresponding reaction rates
    rates_X1 = rates[closest_X1_indices_sample]

    # Fixed samples in different branches (sample is the one with median reaction rate)
    sample_idx = np.argsort(rates)[len(rates) // 2]  # index of median value
    x1_sample = torch.tensor(X1_all[sample_idx], dtype=torch.float).unsqueeze(0)  

    
    # === CNN Branch 1 SHAP ===
    
    # 1. Create the wrapper
    wrapper = FullModelWrapperForCNN1(
        full_model=full_model
    )
    
    # 2. Build background and input tensors
    background_cnn1 = torch.tensor(background_cnn1, dtype=torch.float)  
    X1_tensor = torch.tensor(X1_selected, dtype=torch.float)   
    X1_all_tensor = torch.tensor(X1_all, dtype=torch.float)  
    
    # 3. Run SHAP
    explainer = shap.GradientExplainer(wrapper, background_cnn1)
    shap_values_cnn1 = explainer.shap_values(X1_tensor)

    wrapper.eval()
    
    shap_array = shap_values_cnn1
    shap_array = shap_array.squeeze(-1)
    avg_shap = shap_array.mean(axis=0) 

    heatmap_data = avg_shap.T 
    global_vmin = heatmap_data.min()
    global_vmax = heatmap_data.max()

    filtered_heatmap_data = filter_extreme_by_column(heatmap_data, threshold=global_vmax*0.1)

    positions = np.arange(1, 141)
    nucleotides = ["A", "C", "G", "T"]
    feature_names_cnn1 = [f"Pos{pos}_{nt}" for pos in positions for nt in nucleotides]
    
    X1_flat = X1_tensor.view(X1_tensor.size(0), -1).numpy()
    shap_values_cnn1_flat = shap_values_cnn1.squeeze().reshape(X1_tensor.size(0), -1)
    
    segments_cnn1 = [(0, 40), (40, 140)]
    
    for seg_start, seg_end in segments_cnn1:
        seg_feature_indices = list(range(seg_start * 4, seg_end * 4))  # 4 features per position
        seg_feature_names = [feature_names_cnn1[i] for i in seg_feature_indices]
        
        
    segments_cnn1_crRNA = [(0, 40)]
    for seg_start, seg_end in segments_cnn1_crRNA:
        seg_feature_indices = list(range(seg_start * 4, seg_end * 4))  # 4 features per position
        seg_feature_names = [feature_names_cnn1[i] for i in seg_feature_indices]
        
        import pandas as pd
        X_values = X1_flat[:, seg_feature_indices].cpu().numpy() if hasattr(X1_flat[:, seg_feature_indices], "cpu") else X1_flat[:, seg_feature_indices]
        
        df_input = pd.DataFrame(X_values, columns=seg_feature_names)
        df_shap = pd.DataFrame( shap_values_cnn1_flat[:, seg_feature_indices], columns=seg_feature_names)
        
        df_combined = pd.concat([df_input, df_shap], axis=1)

        df_combined.to_csv("indiv_shap_values_cnn1_HT11_11.csv", index=False)

print('finished')

In [ ]:
# Models tested: 11, 13, 14, 30, 31, 33, 76, 77, 88, 95

In [ ]:
import pandas as pd
import csv
from scipy.stats import spearmanr
from collections import Counter

# --- Config ---
file_names = [
    "indiv_shap_values_cnn1_HT11_11.csv", "indiv_shap_values_cnn1_HT11_13.csv",
    "indiv_shap_values_cnn1_HT11_14.csv", "indiv_shap_values_cnn1_HT11_30.csv",
    "indiv_shap_values_cnn1_HT11_31.csv", "indiv_shap_values_cnn1_HT11_33.csv",
    "indiv_shap_values_cnn1_HT11_76.csv", "indiv_shap_values_cnn1_HT11_77.csv",
    "indiv_shap_values_cnn1_HT11_88.csv", "indiv_shap_values_cnn1_HT11_95.csv"
]

results = []

# Step 1: collect results
for file_name in file_names:
    model_id = file_name.split("_")[-1].replace(".csv", "")  # Extract model ID like '27'

    with open(file_name) as f:
        reader = csv.reader(f)
        header = next(reader)

    df = pd.read_csv(file_name, header=None, skiprows=1)
    df.columns = header  # overwrite with true header

    col_counts = Counter(df.columns)
    duplicate_features = [col for col, count in col_counts.items() if count > 1]

    for feature in duplicate_features:
        indices = [i for i, col in enumerate(df.columns) if col == feature]
        for i in range(len(indices)):
            for j in range(i + 1, len(indices)):
                col1 = df.iloc[:, indices[i]]
                col2 = df.iloc[:, indices[j]]
                corr, pval = spearmanr(col1, col2)
                results.append({
                    "Feature": feature,
                    "Model": model_id,
                    "Spearman Correlation": corr,
                    "P-Value": pval
                })

# Step 2: Build DataFrame and order features
df = pd.DataFrame(results)

# Track first appearance order of features
feature_order = []
seen = set()
for row in results:
    feat = row["Feature"]
    if feat not in seen:
        seen.add(feat)
        feature_order.append(feat)

# Use category sorting to preserve order
df["Feature_order"] = pd.Categorical(df["Feature"], categories=feature_order, ordered=True)
df = df.sort_values(by=["Feature_order", "Model"])
df = df.drop(columns=["Feature_order"])

# Step 3: Blank out repeated feature names
def clear_repeated_features(df):
    cleared = []
    current_feat = None
    for _, row in df.iterrows():
        feat = row["Feature"]
        if feat == current_feat:
            row["Feature"] = ""
        else:
            current_feat = feat
        cleared.append(row)
    return pd.DataFrame(cleared)

df_clean = clear_repeated_features(df)

# Step 4: Save CSV
df_clean.to_csv("indiv_spearman_duplicates_grouped_norepeat_HT11_unique_cnn1.csv", index=False)
print("Saved with feature names grouped and de-duplicated to 'indiv_spearman_duplicates_grouped_norepeat_HT11_unique_cnn1.csv'")
